In [1]:
from dotenv import load_dotenv

# Load OPENAI_API_KEY (and any other secrets) from a local .env file.
load_dotenv()


True

# Notebook 3 · Society Multi-Agent System

> **Series:** LangChain MAS Architectures · Travel Agency Use Case

---

## Architecture Overview

In a **society MAS** agents follow explicit *social rules* rather than
authority relationships. Agents evaluate pre-built options and vote publicly.
The group converges when a majority emerges from the public ballot.

```
     ┌──────────────────────────────────────┐
     │          PUBLIC BALLOT  (state)      │
     └────┬──────────┬──────────┬───────────┘
          │          │          │
   ┌──────▼──┐  ┌────▼──┐  ┌───▼──────┐  ┌──────────┐
   │ Budget  │  │Comfort│  │ Culture  │  │Simplicity│
   │Citizen  │  │Citizen│  │ Citizen  │  │ Citizen  │
   └─────────┘  └───────┘  └──────────┘  └──────────┘
          each citizen follows one social rule → public vote
```

## Pattern in this notebook

| LangChain concept | Role in society MAS |
|---|---|
| `AgentState` | Holds `public_board` (votes) + `winning_package` |
| `create_agent` (citizens) | Agents following a social rule, no manager |
| `@tool` + `ToolRuntime` | Citizens read public board, cast vote, update state |
| `Command(update=...)` | Appends ballot entry and updates `public_board` |
| `create_agent` (coordinator) | Runs voting rounds, checks majority, summarises |


## 1 · Setup

In [2]:
# ── Traveler request ─────────────────────────────────────────────────────────
USER_REQUEST = """\
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan""".strip()

# ── Static catalog ────────────────────────────────────────────────────────────
# Agents must only use options listed here.
DESTINATIONS = {
    "Lisbon":    {"best_period": "April–June", "style": "sunny, walkable, relaxed",
                  "notes": "great for food, viewpoints, and compact neighborhoods"},
    "Barcelona": {"best_period": "April–June", "style": "lively, artistic, seaside",
                  "notes": "strong mix of architecture, beach walks, and tapas"},
    "Prague":    {"best_period": "April–June", "style": "historic, compact, lower-cost",
                  "notes": "easy sightseeing with a classic old-town atmosphere"},
}

FLIGHTS = [
    {"destination": "Lisbon",    "code": "TP-833", "price": 180, "type": "direct",  "duration": "3h 05m"},
    {"destination": "Lisbon",    "code": "IB-310", "price": 150, "type": "1 stop",  "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct",  "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop",  "duration": "4h 00m"},
    {"destination": "Prague",    "code": "FR-721", "price": 110, "type": "direct",  "duration": "1h 55m"},
    {"destination": "Prague",    "code": "OS-410", "price": 145, "type": "1 stop",  "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon",    "name": "Baixa Stay",       "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon",    "name": "River Rooms",       "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel",        "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn",        "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague",    "name": "Old Town House",    "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague",    "name": "City Garden Hotel", "price_per_night":  95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon",    "name": "Alfama food walk",                    "tag": "food",    "price": 35},
    {"destination": "Lisbon",    "name": "Belém and river tram day",            "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening",        "tag": "food",    "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Família and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague",    "name": "Old Town walking tour",               "tag": "culture", "price": 18},
    {"destination": "Prague",    "name": "Czech dinner and jazz night",         "tag": "food",    "price": 30},
]

# ── Catalog helpers ───────────────────────────────────────────────────────────
def flights_for(destination: str)    -> list: return [f for f in FLIGHTS    if f["destination"] == destination]
def hotels_for(destination: str)     -> list: return [h for h in HOTELS     if h["destination"] == destination]
def activities_for(destination: str) -> list: return [a for a in ACTIVITIES if a["destination"] == destination]

def catalog_text() -> str:
    lines = []
    for dest, info in DESTINATIONS.items():
        lines.append(f"Destination: {dest}")
        lines.append(f"  Best period : {info['best_period']}")
        lines.append(f"  Style       : {info['style']}")
        lines.append(f"  Notes       : {info['notes']}")
        lines.append("  Flights:")
        for f in flights_for(dest):
            lines.append(f"    - {f['code']} | {f['type']} | EUR {f['price']} | {f['duration']}")
        lines.append("  Hotels:")
        for h in hotels_for(dest):
            lines.append(f"    - {h['name']} | EUR {h['price_per_night']}/night | {h['style']}")
        lines.append("  Activities:")
        for a in activities_for(dest):
            lines.append(f"    - {a['name']} | {a['tag']} | EUR {a['price']}")
        lines.append("")
    return "\n".join(lines).strip()

CATALOG_TEXT = catalog_text()

print("USER_REQUEST:")
print(USER_REQUEST)
print("\nCatalog loaded — destinations:", list(DESTINATIONS.keys()))


USER_REQUEST:
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan

Catalog loaded — destinations: ['Lisbon', 'Barcelona', 'Prague']


## 2 · Candidate Packages & Custom State

### Pre-built packages
Rather than building a plan from scratch, the society evaluates three
pre-built candidate packages — one per destination.
Citizens vote on these fixed options.

### `SocietyState`
Two extra fields beyond the base `AgentState`:

- `public_board` — every vote cast, visible to all citizens.
- `winning_package` — filled once a majority is reached.


In [3]:
from langchain.agents import AgentState
import operator
from typing import Annotated

# -- Candidate packages --
def build_package(destination: str) -> dict:
    # Each package uses the cheapest flight, most affordable hotel,
    # and the first two activities for that destination.
    cheapest_flight = sorted(flights_for(destination), key=lambda f: f["price"])[0]
    cheapest_hotel  = sorted(hotels_for(destination),  key=lambda h: h["price_per_night"])[0]
    return {
        "destination": destination,
        "period":      DESTINATIONS[destination]["best_period"],
        "flight":      cheapest_flight,
        "hotel":       cheapest_hotel,
        "activities":  activities_for(destination)[:2],
    }

PACKAGES = {
    "package_1": build_package("Lisbon"),
    "package_2": build_package("Barcelona"),
    "package_3": build_package("Prague"),
}

def packages_text() -> str:
    lines = []
    for pkg_id, pkg in PACKAGES.items():
        lines.append(f"{pkg_id}: {pkg['destination']}")
        lines.append(f"  Period     : {pkg['period']}")
        lines.append(f"  Flight     : {pkg['flight']['code']} | {pkg['flight']['type']} | EUR {pkg['flight']['price']}")
        lines.append(f"  Hotel      : {pkg['hotel']['name']} | EUR {pkg['hotel']['price_per_night']}/night")
        lines.append("  Activities : " + ", ".join(a["name"] for a in pkg["activities"]))
        lines.append("")
    return "\n".join(lines).strip()

PACKAGES_TEXT = packages_text()

# -- State --
class SocietyState(AgentState):
    # All votes cast so far -- every citizen sees this before voting.
    public_board: Annotated[list[str], operator.add]
    # Set to the winning package_id once majority is reached.
    winning_package: str

print("Packages built:", list(PACKAGES.keys()))


Packages built: ['package_1', 'package_2', 'package_3']


## 3 · Citizen Sub-Agents

Four citizens, each with a different social rule encoded in its system prompt.
Citizens cast a structured vote: `package_id`, `reason`, `confidence`.

> **Society contract:** no manager tells citizens what to do.
> Each citizen follows its own rule and votes publicly.


In [4]:
from langchain.agents import create_agent

# Social rules: one per citizen, never changed at runtime.
CITIZEN_RULES = {
    "budget_citizen":     "Vote for the package with the lowest total trip cost.",
    "comfort_citizen":    "Vote for the package offering the most direct flight and least friction.",
    "culture_citizen":    "Vote for the package with the richest food and cultural activity mix.",
    "simplicity_citizen": "Vote for the most coherent and easy-to-follow 4-day plan.",
}

def make_citizen(rule: str):
    return create_agent(
        model="openai:gpt-4.1-mini",
        tools=[],
        system_prompt=(
            "You are a citizen in a society-style travel MAS.\n"
            "No manager controls you. You follow your social rule and vote publicly.\n"
            f"Your social rule: {rule}\n\n"
            "When asked to vote, reply in this exact format (nothing else):\n"
            "VOTE: <package_1 | package_2 | package_3>\n"
            "REASON: <one sentence>\n"
            "CONFIDENCE: <1-5>"
        ),
    )

budget_citizen     = make_citizen(CITIZEN_RULES["budget_citizen"])
comfort_citizen    = make_citizen(CITIZEN_RULES["comfort_citizen"])
culture_citizen    = make_citizen(CITIZEN_RULES["culture_citizen"])
simplicity_citizen = make_citizen(CITIZEN_RULES["simplicity_citizen"])

print("Citizen sub-agents created:", list(CITIZEN_RULES.keys()))


Citizen sub-agents created: ['budget_citizen', 'comfort_citizen', 'culture_citizen', 'simplicity_citizen']


## 4 · Voting Tools

Each citizen tool follows the same three steps:

1. **Read public board** from `runtime.state["public_board"]`.
2. **Invoke citizen sub-agent** — it sees the request, all packages, and the current board.
3. **Append ballot** to the board via `Command(update=...)`.

> **Society property visible here:** citizens read the *same* public board.
> A late voter can see how earlier voters leaned — transparency is built in.


In [5]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command


def _parse_vote(text: str) -> tuple[str, str, int]:
    """Extract (package_id, reason, confidence) from a citizen's reply."""
    package_id, reason, confidence = "package_1", "no reason given", 3
    for line in text.strip().splitlines():
        if line.upper().startswith("VOTE:"):
            raw = line.split(":", 1)[1].strip().lower()
            if raw in ("package_1", "package_2", "package_3"):
                package_id = raw
        elif line.upper().startswith("REASON:"):
            reason = line.split(":", 1)[1].strip()
        elif line.upper().startswith("CONFIDENCE:"):
            try:
                confidence = int(line.split(":", 1)[1].strip()[0])
            except (ValueError, IndexError):
                pass
    return package_id, reason, confidence


def _make_vote_tool(citizen_name: str, citizen_agent):
    @tool(f"vote_{citizen_name}", description=f"Citizen '{citizen_name}' reads the public board and casts a vote.")
    async def vote_tool(runtime: ToolRuntime) -> str:
        board = runtime.state.get("public_board", [])
        board_text = "\n".join(board) if board else "No votes cast yet."

        response = await citizen_agent.ainvoke({
            "messages": [HumanMessage(content=(
                f"Traveler request:\n{USER_REQUEST}\n\n"
                f"Candidate packages:\n{PACKAGES_TEXT}\n\n"
                f"Public board (all votes so far):\n{board_text}\n\n"
                "Cast your vote following the required format."
            ))]
        })
        reply = response["messages"][-1].content
        pkg_id, reason, conf = _parse_vote(reply)

        entry = f"{citizen_name} -> {pkg_id} (conf {conf}/5): {reason}"
        return Command(update={
            "public_board": [entry],
            "messages": [ToolMessage(entry, tool_call_id=runtime.tool_call_id)],
        })
    return vote_tool


vote_budget     = _make_vote_tool("budget_citizen",     budget_citizen)
vote_comfort    = _make_vote_tool("comfort_citizen",    comfort_citizen)
vote_culture    = _make_vote_tool("culture_citizen",    culture_citizen)
vote_simplicity = _make_vote_tool("simplicity_citizen", simplicity_citizen)

print("Voting tools created.")


Voting tools created.


## 5 · Society Coordinator + Majority Tool

The coordinator runs voting rounds and checks for majority (≥ 3 of 4 citizens
agree). A small `declare_winner` tool writes the winning package to state once
majority is confirmed.


In [6]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command
from collections import Counter


@tool
def declare_winner(winning_package: str, runtime: ToolRuntime) -> str:
    '''Declare the winning package once majority voting is confirmed.'''
    return Command(update={
        "winning_package": winning_package,
        "messages": [ToolMessage(
            f"Winner declared: {winning_package}",
            tool_call_id=runtime.tool_call_id,
        )],
    })


coordinator = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[vote_budget, vote_comfort, vote_culture, vote_simplicity, declare_winner],
    state_schema=SocietyState,
    system_prompt=(
        "You are the orchestrator of a society-style travel MAS.\n"
        "You do NOT vote yourself and you do NOT make any planning decisions.\n\n"
        "Your job:\n"
        "1. Call all four vote tools: vote_budget_citizen, vote_comfort_citizen, "
        "   vote_culture_citizen, vote_simplicity_citizen.\n"
        "2. AFTER EACH ROUND of voting, count the votes from the public_board in state. "
        "   If one package received 3 or more votes, call declare_winner with that package_id.\n"
        "3. If no majority after round 1, run a second round (same order).\n"
        "4. After declaring the winner (or after round 2 if still no majority, "
        "   pick the package with most votes), write a short summary explaining "
        "   which package won and why the citizens converged on it."
    ),
)

print("Society coordinator created.")


Society coordinator created.


## 6 · Run

In [7]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content=USER_REQUEST)],
        "public_board": [],   # no votes yet
        "winning_package": "", # will be set by declare_winner
    }
)


In [8]:
from pprint import pprint
pprint(response)


{'messages': [HumanMessage(content='Plan a 4-day spring trip from Rome.\nRequirements:\n- mid-range budget\n- easy flights\n- central hotel\n- mix of food and culture\n- simple daily plan', additional_kwargs={}, response_metadata={}, id='1e8b2225-2aa4-4ad7-b597-ca74f1f9580b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 363, 'total_tokens': 440, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_14a630111f', 'id': 'chatcmpl-DPqn93WXqJYzAypbEItWBFkD0XXCT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d496e-f90d-7ec0-b1a8-beac6f04f4c3-0', tool_calls=[{'name': 'vote_budget_citizen', 'args': {}

In [9]:
print(response["messages"][-1].content)


The winning package for the 4-day spring trip from Rome is package_3. This package was favored by three out of the four citizens: the budget, comfort, and simplicity citizens. They highlighted that package_3 offers the best alignment with the mid-range budget, includes easy direct flights, and provides a central hotel with a good balance of food and culture. The plan is also straightforward and easy to follow, making it an ideal choice. The culture citizen preferred a different package for a richer cultural experience but the majority converged on package_3 for its overall practicality and comfort.


## 7 · Key Takeaways

| What you saw | Why it matters |
|---|---|
| `SocietyState` with `public_board` | All votes fully visible to every subsequent voter |
| Citizens read the board before voting | Later voters can see emerging consensus |
| `declare_winner` updates `winning_package` | Clean state transition once majority is confirmed |
| Coordinator counts votes but does not vote | The *society* decides, not a manager |
